In [80]:
import numpy as np
import pandas as pd
from scipy.stats import chisquare, ttest_ind
from statsmodels.stats.proportion import proportions_ztest

In [81]:
#Load the users level table
users = pd.read_csv('../data/processed/user_level_analysis.csv')

#Inspect
users.head()

#Verify all user_ids are unique
assert users['user_id'].is_unique, 'User IDs are not unique'

#verify two variants and two segments
users['variant'].unique()
users['segment'].unique()

#Verify no null values
users.isna().sum()

#Inspect all numeric columns
users.describe()

,total_sessions,total_completed_sessions,total_abandoned_sessions,total_orders,total_spend,total_units
count,13059.000000,13059.000000,13059.000000,13059.000000,13059.000000,13059.000000
mean,2.068152,1.581208,0.486944,1.581208,78.922787,5.965082
std,1.621605,1.377861,0.729471,1.377861,197.419180,11.469525
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,1.000000,16.100000,1.000000
50%,1.000000,1.000000,0.000000,1.000000,33.990000,3.000000
75%,3.000000,2.000000,1.000000,2.000000,71.155000,6.000000
max,14.000000,14.000000,7.000000,14.000000,5244.340000,151.000000


In [82]:
#Chi squared goodness of fit test to verify no Sample Ratio Mismatch.

#Null Hypothesis: Equal distribution of users in treatment and control
#Alternative Hypothesis: Unequal distribution of users in treatment and control

control = users[users['variant'] == 'control']
treatment = users[users['variant'] == 'treatment']
total = len(users)

observed = [len(control), len(treatment)]
expected = [total/2, total/2]

chi2, p_val = chisquare(f_obs=observed, f_exp=expected)

print(f'Chi-square statistic: {chi2:.4f}')
print(f'P-value: {p_val:.4f}') #P-value ~ 0.66.
#p > 0.05. Fail to reject null hypothesis.

Chi-square statistic: 0.1992
P-value: 0.6554


In [83]:
#Load orders level table
orders = pd.read_csv('../data/processed/order_level_analysis.csv')

#Inspect
orders.head()

#Verify two variants
orders['variant'].unique()

#Check all orders are unique
assert orders['order_id'].is_unique, 'Order IDs are not unique'

#Verify no null values
orders.isna().sum()

#Inspect numeric columns
orders.describe()

,order_amount,unit_count
count,20686.000000,20686.000000
mean,49.918329,3.774195
std,146.953452,8.207551
min,3.000000,1.000000
25%,17.190000,1.000000
50%,26.020000,2.000000
75%,39.660000,3.000000
max,4000.000000,79.000000


In [84]:
# Average Order Value

# Inspect aov and spread by variant
aov = orders.groupby('variant').agg(aov=('order_amount', 'mean'), aov_std=('order_amount', 'std'))
print(aov)
# control aov ~ 46.4, treatment aov ~ 53.6
# control order amount stdev ~ 133.84, treatment order amount stdev ~ 159.35

# Null hypothesis: ~ $7.2 gap is not caused by variant
# Alternative hypothesis: ~ $7.2 gap is due to treatment

control_amounts = np.array(orders[orders['variant'] == 'control']['order_amount'])
treatment_amounts = np.array(orders[orders['variant'] == 'treatment']['order_amount'])

t_stat, p_val = ttest_ind(control_amounts, treatment_amounts,equal_var=False)

print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0005
# Reject the null hypothesis.
# Average Order Value went up in treatment from $46.4 to $53.6, a ~ $7.2 increase.

                 aov     aov_std
variant                         
control    46.405776  133.842856
treatment  53.563053  159.346480
T-test statistic: -3.4917
P-value: 0.0005


In [85]:
#Units Per Transaction

#Inspect upt and spread by variant
upt = orders.groupby('variant').agg(upt=('unit_count', 'mean'), upt_std=('unit_count', 'std'))
print(upt)
#control upt ~ 3.48, treatment upt ~ 4.08
#control unit count stdev ~ 7.39, treatment unit count stdev ~ 8.97

#Null hypothesis: ~ 0.6 unit gap is not caused by variant
#Alternative hypothesis: ~ 0.6 unit gap is due to treatment

control_units = np.array(orders[orders['variant'] == 'control']['unit_count'])
treatment_units = np.array(orders[orders['variant'] == 'treatment']['unit_count'])

t_stat, p_val = ttest_ind(control_units, treatment_units, equal_var=False)

print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.000
# Reject the null hypothesis.
# Units Per Transaction went up in treatment from 3.48 to 4.08, a ~0.6 increase.

                upt   upt_std
variant                      
control    3.481583  7.390351
treatment  4.077817  8.967484
T-test statistic: -5.2081
P-value: 0.0000


In [86]:
#Revenue Per Revenue-Generating Users
rpr_users = users[users['total_spend'] != 0]

#Inspect rpru and spread by variant
rpru = rpr_users.groupby('variant').agg(rpru=('total_spend', 'mean'), rpru_std=('total_spend', 'std'))
#control rpru ~ 85.63, treatment rpru ~ 97.72
#control unit count stdev ~ 191.62, treatment unit count stdev ~ 227.06
print(rpru)

#Null hypothesis: ~ $12.09 gap is not caused by variant
#Alternative hypothesis: ~ $12.09 gap is due to treatment

control_spend = np.array(rpr_users[rpr_users['variant'] == 'control']['total_spend'])
treatment_spend = np.array(rpr_users[rpr_users['variant'] == 'treatment']['total_spend'])

t_stat, p_val = ttest_ind(control_spend, treatment_spend, equal_var=False)

print(f'T-test statistic: {t_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p-value ~ 0.0023
# Reject the null hypothesis.
# Revenue per revenue generating user went up in treatment from $85.63 to $97.72, a ~$12.09 increase.


                rpru    rpru_std
variant                         
control    85.632530  191.618912
treatment  97.718869  227.057876
T-test statistic: -3.0477
P-value: 0.0023


In [87]:
# Abandonment Rate
total_control_sessions = control['total_sessions'].sum()
total_control_abandoned_sessions = control['total_abandoned_sessions'].sum()
control_abandonment_rate = total_control_abandoned_sessions/total_control_sessions
print(f'Control abandonment rate = {control_abandonment_rate * 100:.2f}%') #22.18%

total_treatment_sessions = treatment['total_sessions'].sum()
total_treatment_abandoned_sessions = treatment['total_abandoned_sessions'].sum()
treatment_abandonment_rate = total_treatment_abandoned_sessions/total_treatment_sessions
print(f'Treatment abandonment rate = {treatment_abandonment_rate * 100:.2f}%') #24.91%

# Null hypothesis: Abandonment rates across both variants are equal
# Alternative hypothesis: Abandonment rates are different in variants

abandonments = np.array([total_control_abandoned_sessions,total_treatment_abandoned_sessions])
sessions = np.array([total_control_sessions, total_treatment_sessions])

z_stat, p_val = proportions_ztest(count=abandonments, nobs=sessions)

print(f'Z-test statistic: {z_stat:.4f}')
print(f'P-value: {p_val:.4f}') #p~value ~ 0.0000
# Reject null hypothesis. 
# Abandonment rate went up in treatment from 22.18% to 24.91%, a ~12% relative increase.

Control abandonment rate = 22.18%
Treatment abandonment rate = 24.91%
Z-test statistic: -5.2803
P-value: 0.0000


In [88]:
# Frequent vs occasional users
